# 05 · HMM / Viterbi Route Reconstruction

HMM/Viterbi route reconstruction: decoder tuning, final model export, classifier-only vs HMM.

1. Out-of-fold (5-fold GroupKFold) classifier probabilities on TRAIN trips.
2. Grid search of decoder parameters (beta, emission weight, no-path penalty) on TRAIN trips (OOF probs).
3. Fit final classifier on TRAIN, export models/final_model.joblib + models/inference_config.json.
4. Run the exported pipeline on the 10 STUDENT-TEST trips; compare classifier-only vs classifier+HMM.
5. Same comparison for every model from 03_train_compare.py (decoder compatibility).

In [1]:
import sys, time, json, itertools
from pathlib import Path
ROOT = Path.cwd().resolve().parent          # run this notebook from the notebooks/ folder
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import GroupKFold

from pathsnap import config as C
from pathsnap.models import make_model, load_split, subsample_points, X_of, MODEL_SPECS
from pathsnap.candidates import FEATURES
from pathsnap.data_io import load_trip
from pathsnap.hmm import decode_trip, RoadGraph
from pathsnap.metrics import route_continuity, sequence_metrics
from pathsnap.pipeline import Predictor

In [2]:
MODEL = "XGBoost"     # model selected in notebook 03
bf = C.RESULTS / f"optuna_best_{MODEL}.json"
PARAMS = json.load(open(bf))["best_params"] if bf.exists() else {}
print("model", MODEL, "params", PARAMS or "(defaults)", flush=True)

model XGBoost params {'n_estimators': 700, 'learning_rate': 0.05760345546210345, 'max_depth': 6, 'min_child_weight': 0.514935123851333, 'subsample': 0.8075229530089713, 'colsample_bytree': 0.7109394684006103, 'reg_lambda': 0.0033532576310608196}


In [3]:
def point_acc(pred, ref, rel):
    return float((pred == ref)[rel].mean())

In [4]:
def cls_from(c, n):
    out = np.full(n, np.nan)
    if len(c):
        top = c.loc[c.groupby("pt").p.idxmax()]
        out[top.pt.to_numpy()] = top.arc.to_numpy()
    return out

In [5]:
rng = np.random.default_rng(C.SEED)
tr_c, tr_p = load_split("train")
sub_c, _ = subsample_points(tr_c, tr_p)
all_tr_c = pd.read_pickle(C.PROCESSED / "train_cands.pkl")
all_tr_p = pd.read_pickle(C.PROCESSED / "train_points.pkl")

### 1. OOF probabilities for every candidate of every TRAIN point

In [6]:
trips = np.array(sorted(all_tr_c.trip.unique()))
oof = []
for a, b in GroupKFold(n_splits=5).split(trips, groups=trips):
    tr_trips, va_trips = set(trips[a]), set(trips[b])
    fit_c = sub_c[sub_c.trip.isin(tr_trips)]
    m = make_model(MODEL, PARAMS); m.fit(X_of(fit_c), fit_c.label.to_numpy())
    va = all_tr_c[all_tr_c.trip.isin(va_trips)]
    oof.append(va[["trip", "pt", "arc", "t_proj", "label"]].assign(p=m.predict_proba(X_of(va))[:, 1]))
oof = pd.concat(oof)

### 2. decoder grid search on a sample of TRAIN trips

In [7]:
sizes = all_tr_p.groupby("trip").size()
tune_trips = sorted(rng.choice(sizes[sizes <= 5000].index.to_numpy(), 30, replace=False))
cache = {}
for tid in tune_trips:
    trip = load_trip(C.TRAIN_DIR / tid, with_route=False)
    P = all_tr_p[all_tr_p.trip == tid].sort_values("pt")
    cache[tid] = (trip, RoadGraph(trip), oof[oof.trip == tid], P.ref_arc.to_numpy(),
                  (P.ref_dist <= C.LABEL_RELIABLE_M).to_numpy())
grid = []
base = np.mean([point_acc(cls_from(c, len(t.gps_xy)), r, rel) for t, g, c, r, rel in cache.values()])
print(f"classifier-only OOF top1 on tuning trips: {base:.4f}", flush=True)
for beta, w_e, nop in itertools.product([2, 5, 10, 20, 50], [0.5, 1.0, 2.0], [-10.0, -30.0]):
    t0 = time.time(); accs, conts = [], []
    for trip, g, c, ref, rel in cache.values():
        out = decode_trip(trip, c, beta=beta, w_e=w_e, no_path_logp=nop, graph=g)
        accs.append(point_acc(out, ref, rel)); conts.append(route_continuity(out, trip.arcs))
    grid.append(dict(beta=beta, w_e=w_e, no_path_logp=nop, train_oof_top1=np.mean(accs),
                     train_oof_continuity=np.nanmean(conts), seconds=time.time() - t0))
    print(grid[-1], flush=True)
grid = pd.DataFrame(grid).sort_values("train_oof_top1", ascending=False)
grid.to_csv(C.RESULTS / "hmm_decoder_grid.csv", index=False)
best = grid.iloc[0]
dec = dict(beta=float(best.beta), w_e=float(best.w_e), no_path_logp=float(best.no_path_logp),
           gap_s=C.GAP_BREAK_S)
print("best decoder", dec, flush=True)

classifier-only OOF top1 on tuning trips: 0.9206


{'beta': 2, 'w_e': 0.5, 'no_path_logp': -10.0, 'train_oof_top1': np.float64(0.9411683750644), 'train_oof_continuity': np.float64(0.9701725000204393), 'seconds': 8.836642980575562}


{'beta': 2, 'w_e': 0.5, 'no_path_logp': -30.0, 'train_oof_top1': np.float64(0.943815803866287), 'train_oof_continuity': np.float64(0.9707206313936637), 'seconds': 3.148015022277832}


{'beta': 2, 'w_e': 1.0, 'no_path_logp': -10.0, 'train_oof_top1': np.float64(0.9453889106504626), 'train_oof_continuity': np.float64(0.9713224540029393), 'seconds': 3.2590537071228027}


{'beta': 2, 'w_e': 1.0, 'no_path_logp': -30.0, 'train_oof_top1': np.float64(0.9471468062159161), 'train_oof_continuity': np.float64(0.9718716516142034), 'seconds': 3.131439208984375}


{'beta': 2, 'w_e': 2.0, 'no_path_logp': -10.0, 'train_oof_top1': np.float64(0.9443465496503592), 'train_oof_continuity': np.float64(0.971680874100507), 'seconds': 3.56677508354187}


... (22 outputs left out)

{'beta': 50, 'w_e': 1.0, 'no_path_logp': -30.0, 'train_oof_top1': np.float64(0.9393955005221433), 'train_oof_continuity': np.float64(0.9674837058563728), 'seconds': 3.1569628715515137}


{'beta': 50, 'w_e': 2.0, 'no_path_logp': -10.0, 'train_oof_top1': np.float64(0.9346821668499496), 'train_oof_continuity': np.float64(0.964095413030398), 'seconds': 3.0356109142303467}


{'beta': 50, 'w_e': 2.0, 'no_path_logp': -30.0, 'train_oof_top1': np.float64(0.936701312511624), 'train_oof_continuity': np.float64(0.9642515930347428), 'seconds': 3.131870985031128}


best decoder {'beta': 5.0, 'w_e': 0.5, 'no_path_logp': -30.0, 'gap_s': 60.0}


### 3. final model export

In [8]:
C.MODELS.mkdir(exist_ok=True)
final = make_model(MODEL, PARAMS); t0 = time.time()
final.fit(X_of(sub_c), sub_c.label.to_numpy()); fit_s = time.time() - t0
joblib.dump(final, C.MODELS / "final_model.joblib")
json.dump(dict(model_name=MODEL, model_file="final_model.joblib", params=PARAMS or MODEL_SPECS[MODEL][1],
               features=FEATURES, cand_radius_m=C.CAND_RADIUS_M, cand_max_k=C.CAND_MAX_K,
               decoder=dec, trained_on="80 train trips (<=1500 labelled points per trip)",
               fit_seconds=fit_s), open(C.MODELS / "inference_config.json", "w"), indent=2)

### 4. student-test: classifier-only vs HMM using the exported pipeline

In [9]:
pred = Predictor()
te_p = pd.read_pickle(C.PROCESSED / "student_test_points.pkl")
rows, keep = [], []
for d in sorted(C.STUDENT_TEST_DIR.iterdir()):
    t0 = time.time(); r = pred.predict_trip_dir(d); total = time.time() - t0
    trip = r["trip"]
    P = te_p[te_p.trip == d.name].sort_values("pt")
    ref, rel = P.ref_arc.to_numpy(), (P.ref_dist <= C.LABEL_RELIABLE_M).to_numpy()
    ref_route = np.loadtxt(d / f"{d.name}.route", dtype=int, ndmin=1)   # scoring only
    cls_route = [int(a) for a in pd.Series(r["classifier_arcs"]).dropna().astype(int).loc[lambda s: s.ne(s.shift())]]
    rows.append(dict(
        trip=d.name, n_points=len(ref), n_labelled=int(rel.sum()),
        cls_top1=point_acc(r["classifier_arcs"], ref, rel), hmm_top1=point_acc(r["hmm_arcs"], ref, rel),
        cls_continuity=route_continuity(r["classifier_arcs"], trip.arcs),
        hmm_continuity=route_continuity(r["hmm_arcs"], trip.arcs),
        **{f"cls_{k}": v for k, v in sequence_metrics(np.array(cls_route, float), None, ref_route).items()},
        **{f"hmm_{k}": v for k, v in sequence_metrics(np.array(r["route_arcs"], float), None, ref_route).items()},
        seconds_total=total, **r["timings"]))
    keep.append(pd.DataFrame({"trip": d.name, "pt": np.arange(len(ref)), "cls_arc": r["classifier_arcs"],
                              "hmm_arc": r["hmm_arcs"], "ref_arc": ref, "labelled": rel}))
    print(rows[-1], flush=True)
per = pd.DataFrame(rows); per.to_csv(C.RESULTS / "hmm_comparison_per_trip.csv", index=False)
pts = pd.concat(keep); pts.to_pickle(C.RESULTS / "preds" / "final_student_test_points.pkl")
lab = pts[pts.labelled]
summary = dict(model=MODEL, decoder=dec,
               pooled_cls_top1=float((lab.cls_arc == lab.ref_arc).mean()),
               pooled_hmm_top1=float((lab.hmm_arc == lab.ref_arc).mean()),
               mean_cls_continuity=per.cls_continuity.mean(), mean_hmm_continuity=per.hmm_continuity.mean(),
               mean_cls_route_f1=per.cls_route_arc_f1.mean(), mean_hmm_route_f1=per.hmm_route_arc_f1.mean(),
               mean_seconds_per_trip=per.seconds_total.mean())
json.dump(summary, open(C.RESULTS / "hmm_summary.json", "w"), indent=2)
print(summary, flush=True)

{'trip': '00000000', 'n_points': 2503, 'n_labelled': 2503, 'cls_top1': 0.9508589692369157, 'hmm_top1': 0.9620455453455853, 'cls_continuity': 0.9776179056754596, 'hmm_continuity': 0.9960031974420464, 'cls_route_arc_precision': 0.8963730569948186, 'cls_route_arc_recall': 0.9664804469273743, 'cls_route_arc_f1': 0.9301075268817204, 'hmm_route_arc_precision': 0.9030612244897959, 'hmm_route_arc_recall': 0.9888268156424581, 'hmm_route_arc_f1': 0.944, 'seconds_total': 0.5159995555877686, 'candidates_features_s': 0.24419784545898438, 'classifier_s': 0.06840920448303223, 'hmm_s': 0.1313011646270752, 'stitch_s': 0.06543374061584473}


{'trip': '00000007', 'n_points': 1354, 'n_labelled': 1354, 'cls_top1': 0.96602658788774, 'hmm_top1': 1.0, 'cls_continuity': 0.9985218033998522, 'hmm_continuity': 1.0, 'cls_route_arc_precision': 0.983739837398374, 'cls_route_arc_recall': 0.983739837398374, 'cls_route_arc_f1': 0.983739837398374, 'hmm_route_arc_precision': 1.0, 'hmm_route_arc_recall': 0.983739837398374, 'hmm_route_arc_f1': 0.9918032786885246, 'seconds_total': 0.38794946670532227, 'candidates_features_s': 0.08507752418518066, 'classifier_s': 0.029578208923339844, 'hmm_s': 0.05483412742614746, 'stitch_s': 0.2141282558441162}


{'trip': '00000009', 'n_points': 1049, 'n_labelled': 1049, 'cls_top1': 0.8856053384175405, 'hmm_top1': 0.9180171591992373, 'cls_continuity': 0.8998091603053435, 'hmm_continuity': 0.950381679389313, 'cls_route_arc_precision': 0.8927680798004988, 'cls_route_arc_recall': 0.8344988344988346, 'cls_route_arc_f1': 0.8626506024096385, 'hmm_route_arc_precision': 0.9484304932735426, 'hmm_route_arc_recall': 0.986013986013986, 'hmm_route_arc_f1': 0.9668571428571429, 'seconds_total': 0.672898530960083, 'candidates_features_s': 0.1636512279510498, 'classifier_s': 0.05819344520568848, 'hmm_s': 0.4103865623474121, 'stitch_s': 0.03288435935974121}


{'trip': '00000040', 'n_points': 2995, 'n_labelled': 2995, 'cls_top1': 0.9572621035058431, 'hmm_top1': 0.9816360601001669, 'cls_continuity': 0.9686038744154977, 'hmm_continuity': 0.9959919839679359, 'cls_route_arc_precision': 0.9245283018867925, 'cls_route_arc_recall': 0.9514563106796117, 'cls_route_arc_f1': 0.9377990430622011, 'hmm_route_arc_precision': 0.9935691318327974, 'hmm_route_arc_recall': 1.0, 'hmm_route_arc_f1': 0.996774193548387, 'seconds_total': 0.6901845932006836, 'candidates_features_s': 0.3095369338989258, 'classifier_s': 0.07056999206542969, 'hmm_s': 0.20410633087158203, 'stitch_s': 0.09492301940917969}


{'trip': '00000052', 'n_points': 262, 'n_labelled': 262, 'cls_top1': 0.9885496183206107, 'hmm_top1': 1.0, 'cls_continuity': 0.9501915708812261, 'hmm_continuity': 0.9616858237547893, 'cls_route_arc_precision': 0.963855421686747, 'cls_route_arc_recall': 0.8421052631578947, 'cls_route_arc_f1': 0.898876404494382, 'hmm_route_arc_precision': 1.0, 'hmm_route_arc_recall': 0.9789473684210527, 'hmm_route_arc_f1': 0.9893617021276596, 'seconds_total': 0.10194206237792969, 'candidates_features_s': 0.042609214782714844, 'classifier_s': 0.011868953704833984, 'hmm_s': 0.036042213439941406, 'stitch_s': 0.008047342300415039}


{'trip': '00000056', 'n_points': 3958, 'n_labelled': 3931, 'cls_top1': 0.9249554820656322, 'hmm_top1': 0.9687102518443145, 'cls_continuity': 0.9183725044225424, 'hmm_continuity': 0.9668941117007834, 'cls_route_arc_precision': 0.9431137724550899, 'cls_route_arc_recall': 0.8848314606741573, 'cls_route_arc_f1': 0.9130434782608696, 'hmm_route_arc_precision': 0.8970464135021097, 'hmm_route_arc_recall': 0.9953183520599251, 'hmm_route_arc_f1': 0.9436307146027519, 'seconds_total': 1.3668193817138672, 'candidates_features_s': 0.33312129974365234, 'classifier_s': 0.2687685489654541, 'hmm_s': 0.6648802757263184, 'stitch_s': 0.09023213386535645}


{'trip': '00000065', 'n_points': 768, 'n_labelled': 768, 'cls_top1': 0.8736979166666666, 'hmm_top1': 0.8984375, 'cls_continuity': 0.9569752281616688, 'hmm_continuity': 0.9856584093872229, 'cls_route_arc_precision': 0.9097222222222222, 'cls_route_arc_recall': 0.903448275862069, 'cls_route_arc_f1': 0.9065743944636678, 'hmm_route_arc_precision': 0.9863945578231292, 'hmm_route_arc_recall': 1.0, 'hmm_route_arc_f1': 0.9931506849315068, 'seconds_total': 0.5935735702514648, 'candidates_features_s': 0.24885129928588867, 'classifier_s': 0.05209088325500488, 'hmm_s': 0.24199128150939941, 'stitch_s': 0.04537487030029297}


{'trip': '00000076', 'n_points': 1486, 'n_labelled': 1486, 'cls_top1': 0.9347240915208613, 'hmm_top1': 0.9851951547779273, 'cls_continuity': 0.9212121212121213, 'hmm_continuity': 0.9750841750841751, 'cls_route_arc_precision': 0.8793565683646113, 'cls_route_arc_recall': 0.8770053475935828, 'cls_route_arc_f1': 0.8781793842034805, 'hmm_route_arc_precision': 0.9367088607594937, 'hmm_route_arc_recall': 0.9893048128342246, 'hmm_route_arc_f1': 0.9622886866059817, 'seconds_total': 1.640700101852417, 'candidates_features_s': 0.6120738983154297, 'classifier_s': 0.07510614395141602, 'hmm_s': 0.7703914642333984, 'stitch_s': 0.16903281211853027}


{'trip': '00000077', 'n_points': 3911, 'n_labelled': 3911, 'cls_top1': 0.8084888775249297, 'hmm_top1': 0.8918435182817693, 'cls_continuity': 0.9421994884910486, 'hmm_continuity': 0.9948849104859335, 'cls_route_arc_precision': 0.8301282051282052, 'cls_route_arc_recall': 0.9664179104477612, 'cls_route_arc_f1': 0.8931034482758621, 'hmm_route_arc_precision': 0.9305555555555556, 'hmm_route_arc_recall': 1.0, 'hmm_route_arc_f1': 0.9640287769784173, 'seconds_total': 1.0261428356170654, 'candidates_features_s': 0.21839380264282227, 'classifier_s': 0.16694378852844238, 'hmm_s': 0.507479190826416, 'stitch_s': 0.125870943069458}


{'trip': '00000091', 'n_points': 874, 'n_labelled': 874, 'cls_top1': 0.9096109839816934, 'hmm_top1': 0.9588100686498856, 'cls_continuity': 0.9152348224513173, 'hmm_continuity': 0.9713631156930126, 'cls_route_arc_precision': 0.9242424242424242, 'cls_route_arc_recall': 0.7887931034482759, 'cls_route_arc_f1': 0.8511627906976744, 'hmm_route_arc_precision': 0.9914163090128756, 'hmm_route_arc_recall': 0.9956896551724138, 'hmm_route_arc_f1': 0.9935483870967742, 'seconds_total': 0.39788198471069336, 'candidates_features_s': 0.12479376792907715, 'classifier_s': 0.04913210868835449, 'hmm_s': 0.18783140182495117, 'stitch_s': 0.031236886978149414}


{'model': 'XGBoost', 'decoder': {'beta': 5.0, 'w_e': 0.5, 'no_path_logp': -30.0, 'gap_s': 60.0}, 'pooled_cls_top1': 0.909214446244708, 'pooled_hmm_top1': 0.9520200700360634, 'mean_cls_continuity': np.float64(0.9448738479416077), 'mean_hmm_continuity': np.float64(0.9797947406905212), 'mean_cls_route_f1': np.float64(0.905523691014787), 'mean_hmm_route_f1': np.float64(0.9745443567437146), 'mean_seconds_per_trip': np.float64(0.7394092082977295)}


### 5. decoder on top of every compared model (student test)

In [10]:
all_te = pd.read_pickle(C.PROCESSED / "student_test_cands.pkl")
rows = []
for name in MODEL_SPECS:
    f = C.RESULTS / "preds" / f"test_scores_{name}.pkl"
    if not f.exists():
        continue
    s = pd.read_pickle(f)
    c = all_te.merge(s[["trip", "pt", "arc", "p"]], on=["trip", "pt", "arc"])
    cl, hm, ct = [], [], []
    for tid, g in c.groupby("trip"):
        trip = load_trip(C.STUDENT_TEST_DIR / tid, with_route=False)
        P = te_p[te_p.trip == tid].sort_values("pt")
        ref, rel = P.ref_arc.to_numpy(), (P.ref_dist <= C.LABEL_RELIABLE_M).to_numpy()
        out = decode_trip(trip, g, **{k: dec[k] for k in ("beta", "w_e", "no_path_logp")}, gap_s=dec["gap_s"])
        cls = cls_from(g, len(ref))
        cl.append((cls == ref)[rel]); hm.append((out == ref)[rel]); ct.append(route_continuity(out, trip.arcs))
    rows.append(dict(model=name, cls_top1=np.concatenate(cl).mean(), hmm_top1=np.concatenate(hm).mean(),
                     hmm_continuity=np.nanmean(ct)))
    print(rows[-1], flush=True)
pd.DataFrame(rows).to_csv(C.RESULTS / "hmm_all_models.csv", index=False)

{'model': 'LogisticRegression', 'cls_top1': np.float64(0.8715831286259342), 'hmm_top1': np.float64(0.9190926671196362), 'hmm_continuity': np.float64(0.9787851935375038)}


{'model': 'GaussianNB', 'cls_top1': np.float64(0.8478544922385407), 'hmm_top1': np.float64(0.9048241258558511), 'hmm_continuity': np.float64(0.9769504909250882)}


{'model': 'KNN', 'cls_top1': np.float64(0.8751371975121518), 'hmm_top1': np.float64(0.9283436993675848), 'hmm_continuity': np.float64(0.9771223703642453)}


{'model': 'DecisionTree', 'cls_top1': np.float64(0.8903465217164062), 'hmm_top1': np.float64(0.9447028693879684), 'hmm_continuity': np.float64(0.9781908296444388)}


{'model': 'RandomForest', 'cls_top1': np.float64(0.896252547953797), 'hmm_top1': np.float64(0.9444938065123086), 'hmm_continuity': np.float64(0.9794530183956498)}


{'model': 'LightGBM', 'cls_top1': np.float64(0.9052422516071709), 'hmm_top1': np.float64(0.9505566299064444), 'hmm_continuity': np.float64(0.979579940537929)}


{'model': 'XGBoost', 'cls_top1': np.float64(0.9072283489259395), 'hmm_top1': np.float64(0.9488318611822506), 'hmm_continuity': np.float64(0.9797320743353988)}


{'model': 'MLP', 'cls_top1': np.float64(0.9034129514451471), 'hmm_top1': np.float64(0.9374902001777035), 'hmm_continuity': np.float64(0.979336969727914)}
